# Notebook 04 — Fine-tuning du bi-encoder

On fine-tune `bge-small-en-v1.5` sur le dataset ATS pour qu'il apprenne à scorer
des paires CV/offre de manière plus précise.
On utilise `CosineSimilarityLoss` avec des scores continus normalisés entre 0 et 1.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics.pairwise import cosine_similarity

from src.data_utils import load_dataset_fit, group_split
from src.evaluation import classify, find_best_thresholds, compute_f1

## Chargement du dataset ATS

Ce dataset contient ~6300 paires CV/offre avec des scores ATS continus (entre 19 et 90).
On normalise les scores entre 0 et 1 pour les utiliser avec CosineSimilarityLoss.

In [2]:
print("Chargement du dataset principal...")
df_ats = load_dataset_fit()
print(f"\nAperçu des labels:")
print(df_ats['label'].value_counts())

Chargement du dataset principal...
Dataset chargé: 8000 paires
Colonnes: ['resume_text', 'job_description_text', 'label']

Aperçu des labels:
label
No Fit           4000
Potential Fit    2000
Good Fit         2000
Name: count, dtype: int64


In [3]:
# on regarde à quoi ressemblent les colonnes
print(df_ats.head(3).to_string())

# trouver les colonnes CV, offre, score
# TODO: adapter les noms de colonnes selon ce qu'on trouve dans df_ats.columns

## Préparation des données

On normalise les scores entre 0 et 1, puis on fait un group split par CV.

In [4]:
cv_col = 'resume_text'
job_col = 'job_description_text'

# mapper les labels en scores continus pour CosineSimilarityLoss
label_to_score = {'Good Fit': 1.0, 'Potential Fit': 0.5, 'No Fit': 0.0}
df_ats['score_norm'] = df_ats['label'].map(label_to_score)

print(f"Distribution des scores normalisés:")
print(df_ats['score_norm'].value_counts())
print(f"Score normalisé — moyenne: {df_ats['score_norm'].mean():.4f}")

# group split par CV pour éviter le data leakage
train_ats, test_ats = group_split(df_ats)

Distribution des scores normalisés:
score_norm
0.0    4000
0.5    2000
1.0    2000
Name: count, dtype: int64
Score normalisé — moyenne: 0.3750
CVs uniques: 643 pour 8000 paires
Train: 6595 | Test: 1405
Overlap CVs train/test: 0 (doit être 0)


## Préparation des InputExamples pour sentence-transformers

In [5]:
# créer les exemples d'entraînement
train_examples = [
    InputExample(
        texts=[row[cv_col], row[job_col]],
        label=float(row['score_norm'])
    )
    for _, row in train_ats.iterrows()
]

print(f"Exemples d'entraînement: {len(train_examples)}")
print(f"Exemple 0 — score: {train_examples[0].label:.4f}")

Exemples d'entraînement: 6595
Exemple 0 — score: 0.0000


## Fine-tuning avec CosineSimilarityLoss

Hyperparamètres :
- Learning rate : 2e-5
- Batch size : 8 (réduit depuis 16 pour tenir en 4GB VRAM)
- Epochs : 3
- fp16 : activé pour tenir en 4GB VRAM

In [6]:
# vider le cache GPU avant de charger le modèle
import torch
torch.cuda.empty_cache()

# charger le modèle de base
model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# dataloader — batch 8 pour tenir en 4GB VRAM
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

# loss function
train_loss = losses.CosineSimilarityLoss(model)

# evaluator sur le test set ATS
test_examples = [
    InputExample(texts=[row[cv_col], row[job_col]], label=float(row['score_norm']))
    for _, row in test_ats.iterrows()
]
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(test_examples, name='ats-test')

# fine-tuning
print("Début du fine-tuning...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=3,
    warmup_steps=100,
    output_path='../models/bge-small-finetuned',
    use_amp=True,  # fp16
    show_progress_bar=True,
    evaluation_steps=500,
    save_best_model=True,
)

print("Fine-tuning terminé !")
print("Modèle sauvegardé dans models/bge-small-finetuned/")

Début du fine-tuning...


/home/alexandre/Epitech/UVT/DeepLearning/CV-matching/.venv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:1011: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

Iteration:   0%|          | 0/825 [00:00<?, ?it/s]

Iteration:   0%|          | 0/825 [00:00<?, ?it/s]

Iteration:   0%|          | 0/825 [00:00<?, ?it/s]

Fine-tuning terminé !
Modèle sauvegardé dans models/bge-small-finetuned/


## Évaluation du modèle fine-tuné sur le dataset principal

On charge le dataset principal (notebook 01) et on compare le F1 avant/après fine-tuning.

In [7]:
# charger le modèle fine-tuné
model_ft = SentenceTransformer('../models/bge-small-finetuned')
print("Modèle fine-tuné chargé")

Modèle fine-tuné chargé


In [8]:
# charger le dataset principal
df_main = load_dataset_fit()
train_main, test_main = group_split(df_main)

# encoder avec le modèle fine-tuné
print("Encodage avec modèle fine-tuné...")
cv_ft = model_ft.encode(test_main['resume_text'].tolist(), show_progress_bar=True, batch_size=32)
job_ft = model_ft.encode(test_main['job_description_text'].tolist(), show_progress_bar=True, batch_size=32)

scores_ft = [
    cosine_similarity([cv_ft[i]], [job_ft[i]])[0][0]
    for i in range(len(test_main))
]

# calibrer les seuils sur le train
cv_train_ft = model_ft.encode(train_main['resume_text'].tolist(), show_progress_bar=True, batch_size=32)
job_train_ft = model_ft.encode(train_main['job_description_text'].tolist(), show_progress_bar=True, batch_size=32)
scores_train_ft = [
    cosine_similarity([cv_train_ft[i]], [job_train_ft[i]])[0][0]
    for i in range(len(train_main))
]

high_ft, low_ft, _ = find_best_thresholds(scores_train_ft, train_main['label'].tolist())
f1_ft = compute_f1(scores_ft, test_main['label'].tolist(), high_ft, low_ft)

print(f"\nF1 weighted — bge-small fine-tuné: {f1_ft:.4f}")
print(f"Seuils: Good Fit >= {high_ft:.2f}, Potential Fit >= {low_ft:.2f}")

Dataset chargé: 8000 paires
Colonnes: ['resume_text', 'job_description_text', 'label']
CVs uniques: 643 pour 8000 paires
Train: 6595 | Test: 1405
Overlap CVs train/test: 0 (doit être 0)
Encodage avec modèle fine-tuné...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Batches:   0%|          | 0/207 [00:00<?, ?it/s]

Batches:   0%|          | 0/207 [00:00<?, ?it/s]


F1 weighted — bge-small fine-tuné: 0.6219
Seuils: Good Fit >= 0.65, Potential Fit >= 0.30


## Récapitulatif

In [ ]:
print("=" * 50)
print("RÉSULTATS — FINE-TUNING bge-small")
print("=" * 50)
print(f"Dataset utilisé: {len(train_ats)} paires (train)")
print(f"Epochs: 3 | LR: 2e-5 | Batch: 8 | fp16: True")
print(f"\nF1 bge-small pré-entraîné (troncation): voir notebook 03")
print(f"F1 bge-small fine-tuné: {f1_ft:.4f}")
print("\nModèle sauvegardé: models/bge-small-finetuned/")

RÉSULTATS — FINE-TUNING bge-small
Dataset utilisé: 6595 paires (train)
Epochs: 3 | LR: 2e-5 | Batch: 8 | fp16: True

F1 bge-small pré-entraîné (troncation): voir notebook 03
F1 bge-small fine-tuné: 0.6219

Modèle sauvegardé: models/bge-small-finetuned/
